In [ ]:
import os
import sys
import math
from typing import Dict, List, Tuple, Optional

import psycopg

In [ ]:

# ---- Connection helper from your snippet ----
def _connect() -> psycopg.Connection:
    """Create a new psycopg (v3) connection using env vars or sensible defaults."""
    host = os.getenv("MUSICBRAINZ_DB_HOST", "localhost")
    port = int(os.getenv("MUSICBRAINZ_DB_PORT", "5432"))
    dbname = os.getenv("MUSICBRAINZ_DB_NAME", "musicbrainz_db")
    user = os.getenv("MUSICBRAINZ_DB_USER", "musicbrainz")
    password = os.getenv("MUSICBRAINZ_DB_PASSWORD", "musicbrainz")

    return psycopg.connect(
        host=host,
        port=port,
        dbname=dbname,
        user=user,
        password=password,
    )

# ---------------- Utility ----------------
TARGET_TOTAL = 1_000_000
FETCH_CHUNK = 5_000  # streaming chunk size

def table_exists(cur: psycopg.Cursor, table: str) -> bool:
    cur.execute("""
        SELECT 1
        FROM information_schema.tables
        WHERE table_schema IN ('public', 'musicbrainz')
          AND table_name = %s
        LIMIT 1
    """, (table,))
    return cur.fetchone() is not None

def column_exists(cur: psycopg.Cursor, table: str, column: str) -> bool:
    cur.execute("""
        SELECT 1
        FROM information_schema.columns
        WHERE table_name = %s AND column_name = %s
        LIMIT 1
    """, (table, column))
    return cur.fetchone() is not None

def compute_quotas(total: int, buckets: List[Tuple[int, str, int]]) -> Dict[int, int]:
    """
    buckets: list of (genre_id, genre_name, count_in_genre).
    Evenly divide 'total' across available buckets (only those with count > 0).
    """
    nonempty = [(gid, name, c) for gid, name, c in buckets if c > 0]
    if not nonempty:
        return {}
    g = len(nonempty)
    base = total // g
    rem = total % g

    quotas = {}
    # To improve odds of filling, give remainder to larger buckets first
    nonempty_sorted = sorted(nonempty, key=lambda x: x[2], reverse=True)
    for i, (gid, _name, count) in enumerate(nonempty_sorted):
        want = base + (1 if i < rem else 0)
        quotas[gid] = min(want, count)  # can't exceed available
    # If rounding capped some small genres, redistribute deficit later (simple pass)
    deficit = total - sum(quotas.values())
    if deficit > 0:
        for gid, _name, count in nonempty_sorted:
            if quotas[gid] < count:
                take = min(deficit, count - quotas[gid])
                quotas[gid] += take
                deficit -= take
                if deficit == 0:
                    break
    return quotas

def main():
    with _connect() as conn, conn.cursor() as cur, open("million_titles.txt", "w", encoding="utf-8") as outfile:
        with conn.cursor() as cur:
            # Detect the best available genre mapping
            has_genre = table_exists(cur, "genre")
            use_official = False
            link_table = None

            if has_genre:
                if table_exists(cur, "l_recording_genre"):
                    link_table = "l_recording_genre"
                    # Heuristic: columns are likely (recording, genre) OR (entity, genre)
                    # We'll try both later.
                    use_official = True
                elif table_exists(cur, "recording_genre"):
                    link_table = "recording_genre"
                    use_official = True

            has_rec_tag = table_exists(cur, "recording_tag") and table_exists(cur, "tag")
            has_is_genre = has_rec_tag and column_exists(cur, "tag", "is_genre")

            # 1) Get genre buckets (id, name, count)
            buckets: List[Tuple[int, str, int]] = []

            if use_official:
                # Try common column combos for the link table
                tried_sql = []
                sqls = [
                    # l_recording_genre with columns recording, genre
                    f"""
                    SELECT g.id, g.name, COUNT(*) AS n
                    FROM genre g
                    JOIN {link_table} lrg ON lrg.genre = g.id
                    JOIN recording r ON r.id = lrg.recording
                    GROUP BY g.id, g.name
                    """,
                    # l_recording_genre with entity column
                    f"""
                    SELECT g.id, g.name, COUNT(*) AS n
                    FROM genre g
                    JOIN {link_table} lrg ON lrg.genre = g.id
                    JOIN recording r ON r.id = lrg.entity
                    GROUP BY g.id, g.name
                    """,
                ]
                got = False
                for sql in sqls:
                    try:
                        cur.execute(sql)
                        rows = cur.fetchall()
                        if rows:
                            buckets = [(int(r[0]), r[1], int(r[2])) for r in rows]
                            got = True
                            break
                    except Exception as e:
                        tried_sql.append((sql, str(e)))
                        conn.rollback()
                if not got and has_rec_tag:
                    # Fallback: genre table exists but link didn't; use tags mapped to official genres by name
                    cur.execute("SELECT id, name FROM genre")
                    gmap = {name.lower(): gid for gid, name in cur.fetchall()}
                    name_filter = tuple(gmap.keys()) if gmap else tuple()

                    if name_filter:
                        where_clause = "LOWER(t.name) = ANY(%s)"
                        cur.execute(f"""
                            SELECT MIN(t.id) AS id, LOWER(t.name) AS name, COUNT(*) AS n
                            FROM tag t
                            JOIN recording_tag rt ON rt.tag = t.id
                            WHERE {where_clause}
                            GROUP BY LOWER(t.name)
                        """, (list(name_filter),))
                        rows = cur.fetchall()
                        buckets = [(gmap[name], name, int(n)) for _id, name, n in rows if name in gmap]
                    else:
                        buckets = []
            elif has_rec_tag:
                # Use tags where tag.is_genre = true if available; else fall back to a small curated list
                if has_is_genre:
                    cur.execute("""
                        SELECT t.id, t.name, COUNT(*) AS n
                        FROM tag t
                        JOIN recording_tag rt ON rt.tag = t.id
                        WHERE t.is_genre = TRUE
                        GROUP BY t.id, t.name
                    """)
                    rows = cur.fetchall()
                    buckets = [(int(i), n, int(c)) for i, n, c in rows]
                else:
                    # Minimal portable list; adjust if you want more/other genres
                    candidate_genres = [
                        "rock","pop","hip hop","electronic","jazz","classical",
                        "metal","folk","country","blues","soul","funk","reggae",
                        "punk","r&b","house","techno","ambient","latin","world"
                    ]
                    cur.execute("""
                        SELECT MIN(t.id) AS id, LOWER(t.name) AS name, COUNT(*) AS n
                        FROM tag t
                        JOIN recording_tag rt ON rt.tag = t.id
                        WHERE LOWER(t.name) = ANY(%s)
                        GROUP BY LOWER(t.name)
                    """, (candidate_genres,))
                    rows = cur.fetchall()
                    buckets = [(int(i), str(nm), int(c)) for i, nm, c in rows]
            else:
                print("No usable genre/recording mapping found (genre/*_genre or recording_tag/tag).", file=sys.stderr)
                sys.exit(2)

            if not buckets:
                print("Found no genre buckets with recordings.", file=sys.stderr)
                sys.exit(2)

            quotas = compute_quotas(TARGET_TOTAL, buckets)
            if not quotas:
                print("Could not compute per-genre quotas.", file=sys.stderr)
                sys.exit(2)

            total_planned = sum(quotas.values())
            if total_planned < TARGET_TOTAL:
                # It's okay—print as many as we can
                print(f"# Planned to output {total_planned} titles (requested {TARGET_TOTAL}); not enough per-genre data.", file=sys.stderr)

            # 2) Stream titles per genre
            out_count = 0

            def stream_titles_official(gid: int, limit: int):
                # Try both link table shapes again, stream by id for stability
                # (Avoid ORDER BY random() for performance.)
                tried_sql_local = []
                for variant in [
                    f"""
                    SELECT r.name
                    FROM recording r
                    JOIN {link_table} lrg ON lrg.recording = r.id
                    WHERE lrg.genre = %s AND r.name IS NOT NULL
                    ORDER BY r.id
                    LIMIT %s
                    """,
                    f"""
                    SELECT r.name
                    FROM recording r
                    JOIN {link_table} lrg ON lrg.entity = r.id
                    WHERE lrg.genre = %s AND r.name IS NOT NULL
                    ORDER BY r.id
                    LIMIT %s
                    """,
                ]:
                    try:
                        cur.execute(variant, (gid, limit))
                        while True:
                            rows = cur.fetchmany(FETCH_CHUNK)
                            if not rows:
                                break
                            for (title,) in rows:
                                yield title
                        return
                    except Exception as e:
                        tried_sql_local.append((variant, str(e)))
                        conn.rollback()
                # If both fail, silently yield nothing for this gid

            def stream_titles_from_tags(tag_id: int, limit: int):
                cur.execute("""
                    SELECT r.name
                    FROM recording r
                    JOIN recording_tag rt ON rt.recording = r.id
                    WHERE rt.tag = %s AND r.name IS NOT NULL
                    ORDER BY r.id
                    LIMIT %s
                """, (tag_id, limit))
                while True:
                    rows = cur.fetchmany(FETCH_CHUNK)
                    if not rows:
                        break
                    for (title,) in rows:
                        yield title

            using_official_stream = use_official or (has_genre and link_table is not None)

            # Map genre name for comments/logging (optional)
            gid_to_name = {gid: name for gid, name, _c in buckets}
            
            for gid, take in quotas.items():
                if take <= 0:
                    continue
                if using_official_stream:
                    gen = stream_titles_official(gid, take)
                else:
                    # In tag mode, gid is actually the tag.id
                    gen = stream_titles_from_tags(gid, take)

                for title in gen:
                    # Print one title per line (UTF-8)
                    if title:
                        outfile.write(title + "\n")
                        out_count += 1
                        if out_count >= TARGET_TOTAL:
                            break
                if out_count >= TARGET_TOTAL:
                    break
